# Imports

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import tensorflow.keras.backend as K

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data = pd.read_csv("/content/drive/MyDrive/CS 224N Project/prelabel_output.csv", index_col=None)
data.head(5)
print(np.shape(data))

(15575, 17)


In [ ]:
data

,Unnamed: 0,speech_id,category,country,phrase,speech,date,word_count,speakerid,lastname,firstname,chamber,state,gender,party,district,nonvoting
0,0,1110000096,adversary,Iran,and nation to pay our bills.,Mr. Speaker. at the beginning of the 110th Con...,2009-01-06,346,111116270,CARDOZA,DENNIS,H,CA,M,D,18.0,voting
1,1,1110000134,ally,Israel,it looks like the days of the old west have re...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting
2,2,1110000134,ally,Israel,innocent nation civilians have been targeted b...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting
3,3,1110000134,ally,Israel,these terrorist outlaws have fired over 8.000 ...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting
4,4,1110000134,ally,Israel,these extremists call for the total destructio...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15570,15570,1110178940,adversary,Russia,and ensuring that our nuclear inspectors are o...,Mr. President. as we end this year. I wanted t...,2010-12-22,2487,111118701,BOXER,BARBARA,S,CA,F,D,NaN,voting
15571,15571,1110178940,adversary,Iran,we enacted tough new sanctions against iranano...,Mr. President. as we end this year. I wanted t...,2010-12-22,2487,111118701,BOXER,BARBARA,S,CA,F,D,NaN,voting
15572,15572,1110178944,adversary,Russia,following the 2008 war with nation.,Mr. President. I rise today to mention a disti...,2010-12-22,346,111117261,KERRY,JOHN,S,MA,M,D,NaN,voting
15573,15573,1110179008,adversary,Russia,more than a year has passed since american ins...,Madam President. when we convened this Congres...,2010-12-22,1555,111120961,REID,HARRY,S,NV,M,D,NaN,voting


In [ ]:
data = data.drop(columns=['Unnamed: 0'])

In [ ]:
data['category'].unique()

array(['adversary', 'ally'], dtype=object)

In [ ]:
data['party'].unique()

array(['D', 'R', 'I'], dtype=object)

In [ ]:
data["is_ally"] = data['category'].apply(lambda x: 1 if x == 'ally' else 0 if x == 'adversary' else None)
data["is_democrat"] = data['party'].apply(lambda x: 1 if x == 'D' else 0 if x == 'R' else None)
data

,speech_id,category,country,phrase,speech,date,word_count,speakerid,lastname,firstname,chamber,state,gender,party,district,nonvoting,is_ally,is_democrat
0,1110000096,adversary,Iran,and nation to pay our bills.,Mr. Speaker. at the beginning of the 110th Con...,2009-01-06,346,111116270,CARDOZA,DENNIS,H,CA,M,D,18.0,voting,0,1.0
1,1110000134,ally,Israel,it looks like the days of the old west have re...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting,1,0.0
2,1110000134,ally,Israel,innocent nation civilians have been targeted b...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting,1,0.0
3,1110000134,ally,Israel,these terrorist outlaws have fired over 8.000 ...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting,1,0.0
4,1110000134,ally,Israel,these extremists call for the total destructio...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15570,1110178940,adversary,Russia,and ensuring that our nuclear inspectors are o...,Mr. President. as we end this year. I wanted t...,2010-12-22,2487,111118701,BOXER,BARBARA,S,CA,F,D,NaN,voting,0,1.0
15571,1110178940,adversary,Iran,we enacted tough new sanctions against iranano...,Mr. President. as we end this year. I wanted t...,2010-12-22,2487,111118701,BOXER,BARBARA,S,CA,F,D,NaN,voting,0,1.0
15572,1110178944,adversary,Russia,following the 2008 war with nation.,Mr. President. I rise today to mention a disti...,2010-12-22,346,111117261,KERRY,JOHN,S,MA,M,D,NaN,voting,0,1.0
15573,1110179008,adversary,Russia,more than a year has passed since american ins...,Madam President. when we convened this Congres...,2010-12-22,1555,111120961,REID,HARRY,S,NV,M,D,NaN,voting,0,1.0


In [ ]:
# removing data from independents
clean_data = data[data['party'] != 'I'].copy()
clean_data["is_democrat"].unique()

array([1., 0.])

In [ ]:
clean_data["is_democrat"] = clean_data["is_democrat"].astype(int)

In [ ]:
len(clean_data)

15393

In [ ]:
clean_data[clean_data.isnull().any(axis=1)]

,speech_id,category,country,phrase,speech,date,word_count,speakerid,lastname,firstname,chamber,state,gender,party,district,nonvoting,is_ally,is_democrat
12,1110000233,ally,Israel,the first will be about nation and the second ...,Mr. President. I rise for a few moments to add...,2009-01-06,1326,111119891,ISAKSON,JOHN,S,GA,M,R,NaN,voting,1,0
13,1110000233,ally,Israel,in november of 2007. i stood at the last natio...,Mr. President. I rise for a few moments to add...,2009-01-06,1326,111119891,ISAKSON,JOHN,S,GA,M,R,NaN,voting,1,0
14,1110000233,ally,Israel,the nation settlement outside gaza.,Mr. President. I rise for a few moments to add...,2009-01-06,1326,111119891,ISAKSON,JOHN,S,GA,M,R,NaN,voting,1,0
15,1110000233,ally,Israel,random attacks coming out of gaza dropping on ...,Mr. President. I rise for a few moments to add...,2009-01-06,1326,111119891,ISAKSON,JOHN,S,GA,M,R,NaN,voting,1,0
16,1110000233,ally,Israel,what nation has done by moving into gaza is a ...,Mr. President. I rise for a few moments to add...,2009-01-06,1326,111119891,ISAKSON,JOHN,S,GA,M,R,NaN,voting,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15570,1110178940,adversary,Russia,and ensuring that our nuclear inspectors are o...,Mr. President. as we end this year. I wanted t...,2010-12-22,2487,111118701,BOXER,BARBARA,S,CA,F,D,NaN,voting,0,1
15571,1110178940,adversary,Iran,we enacted tough new sanctions against iranano...,Mr. President. as we end this year. I wanted t...,2010-12-22,2487,111118701,BOXER,BARBARA,S,CA,F,D,NaN,voting,0,1
15572,1110178944,adversary,Russia,following the 2008 war with nation.,Mr. President. I rise today to mention a disti...,2010-12-22,346,111117261,KERRY,JOHN,S,MA,M,D,NaN,voting,0,1
15573,1110179008,adversary,Russia,more than a year has passed since american ins...,Madam President. when we convened this Congres...,2010-12-22,1555,111120961,REID,HARRY,S,NV,M,D,NaN,voting,0,1


In [ ]:
clean_data

,speech_id,category,country,phrase,speech,date,word_count,speakerid,lastname,firstname,chamber,state,gender,party,district,nonvoting,is_ally,is_democrat
0,1110000096,adversary,Iran,and nation to pay our bills.,Mr. Speaker. at the beginning of the 110th Con...,2009-01-06,346,111116270,CARDOZA,DENNIS,H,CA,M,D,18.0,voting,0,1
1,1110000134,ally,Israel,it looks like the days of the old west have re...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting,1,0
2,1110000134,ally,Israel,innocent nation civilians have been targeted b...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting,1,0
3,1110000134,ally,Israel,these terrorist outlaws have fired over 8.000 ...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting,1,0
4,1110000134,ally,Israel,these extremists call for the total destructio...,Madam Speaker. it looks like the days of the O...,2009-01-06,219,111120850,POE,TED,H,TX,M,R,2.0,voting,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15570,1110178940,adversary,Russia,and ensuring that our nuclear inspectors are o...,Mr. President. as we end this year. I wanted t...,2010-12-22,2487,111118701,BOXER,BARBARA,S,CA,F,D,NaN,voting,0,1
15571,1110178940,adversary,Iran,we enacted tough new sanctions against iranano...,Mr. President. as we end this year. I wanted t...,2010-12-22,2487,111118701,BOXER,BARBARA,S,CA,F,D,NaN,voting,0,1
15572,1110178944,adversary,Russia,following the 2008 war with nation.,Mr. President. I rise today to mention a disti...,2010-12-22,346,111117261,KERRY,JOHN,S,MA,M,D,NaN,voting,0,1
15573,1110179008,adversary,Russia,more than a year has passed since american ins...,Madam President. when we convened this Congres...,2010-12-22,1555,111120961,REID,HARRY,S,NV,M,D,NaN,voting,0,1


In [ ]:
clean_df = clean_data.drop(columns=['district']) # since 1/3 of data has district as NaN
len(clean_df)

15393

In [ ]:
clean_df.isnull().values.any()

False

# Preparing dataset for GPT

In [ ]:
import pandas as pd

# Sample DataFrame
data = {
    "phrase": ["This nation is a vital partner and friend of the United States", "This nation poses a significant threat to global peace"],
    "is_ally": [1, 0],
    "is_democrat": [1, 0]
}

df = pd.DataFrame(data)

In [ ]:
clean_data.columns

Index(['speech_id', 'category', 'country', 'phrase', 'speech', 'date',
       'word_count', 'speakerid', 'lastname', 'firstname', 'chamber', 'state',
       'gender', 'party', 'district', 'nonvoting', 'is_ally', 'is_democrat'],
      dtype='object')

In [ ]:
c = clean_data.copy()

In [ ]:
# idea for later, using speaker id

In [ ]:
all_data_df = c[['phrase', 'is_ally', 'is_democrat']]

In [ ]:
# need to get 100, 200, 300 samples each with balanced ratios

In [ ]:
# fine_tuning df
pos_pos = all_data_df[(all_data_df['is_ally']==1) & (all_data_df['is_democrat']==1)]
pos_neg = all_data_df[(all_data_df['is_ally']==1) & (all_data_df['is_democrat']==0)]
neg_pos = all_data_df[(all_data_df['is_ally']==0) & (all_data_df['is_democrat']==1)]
neg_neg = all_data_df[(all_data_df['is_ally']==0) & (all_data_df['is_democrat']==0)]

In [ ]:
pos_pos_25_chosen = pos_pos.sample(n=25, random_state=1)
pos_pos_50_chosen = pos_pos.sample(n=50, random_state=1)
pos_pos_test_chosen = pos_pos.sample(n=100, random_state=1)

pos_neg_25_chosen = pos_neg.sample(n=25, random_state=1)
pos_neg_50_chosen = pos_neg.sample(n=50, random_state=1)
pos_neg_test_chosen = pos_neg.sample(n=100, random_state=1)

neg_pos_25_chosen = neg_pos.sample(n=25, random_state=1)
neg_pos_50_chosen = neg_pos.sample(n=50, random_state=1)
neg_pos_test_chosen = neg_pos.sample(n=100, random_state=1)

neg_neg_25_chosen = neg_neg.sample(n=25, random_state=1)
neg_neg_50_chosen = neg_neg.sample(n=50, random_state=1)
neg_neg_test_chosen = neg_neg.sample(n=100, random_state=1)

In [ ]:
test_df = pd.concat([pos_pos_test_chosen, pos_neg_test_chosen, neg_pos_test_chosen, neg_neg_test_chosen])

In [ ]:
df = pd.concat([pos_pos_25_chosen, pos_neg_25_chosen, neg_pos_25_chosen, neg_neg_25_chosen])

In [ ]:
df_200_samples = pd.concat([pos_pos_test_chosen, pos_neg_test_chosen, neg_pos_test_chosen, neg_neg_test_chosen])

In [ ]:
df

,phrase,is_ally,is_democrat
1166,maybe nation should wait until this gets to 10...,1,1
6150,it continues to threaten our ally nation.,1,1
11029,a nuclear nation would not only pose a dire th...,1,1
832,my wish continues to be that nation will one d...,1,1
1848,there has been a measure of justice for nation...,1,1
...,...,...,...
14104,president medvedev speaks often and at times e...,0,0
14681,it is important to note that the new start tre...,0,0
5273,the latest information in the last few days is...,0,0
10426,what if nation gets their nuclear weapons beca...,0,0


In [ ]:
# shuffling data
df = df.sample(frac = 1)
df

,phrase,is_ally,is_democrat
3158,the president of the nation medical associatio...,1,0
11403,it has been 1 year since ahmadinejad and his t...,0,1
9448,the effect of this legislation even coming up ...,0,0
10762,since nation instituted its gaza blockade.,1,1
10871,propping up tyrants who lead countries such as...,0,1
...,...,...,...
780,nation had no choice but to retaliate with for...,1,0
14241,new start could pay dividends not only by impr...,0,1
1575,many people have asked why isnt nation respons...,1,1
9557,the united states and the international commun...,0,1


# Preparing Prompt and Completion Pairs

In [ ]:

def create_chat_format(row):
    ally_status = "ally" if row['is_ally'] == 1 else "adversary"
    completion = ally_status

    user_message = f"{row['phrase']} Is the conversation around an ally nation or adversary nation?"

    return {
        "messages": [
            {"role": "system", "content": "You are a smart assistant who can read a phrase and know from the phrase whether Congress is talking about an ally nation or an adversary nation."},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": completion}
        ]
    }


# Convert DataFrame to the required format
chat_data = [create_chat_format(row) for _, row in df.iterrows()]

# Save to a JSONL file
with open('finetuning_data_ally_adv_chat.jsonl', 'w') as f:
    for entry in chat_data:
        f.write(json.dumps(entry) + '\n')

# Need data in chat format

In [ ]:

def create_chat_format(row):
    ally_status = "ally" if row['is_ally'] == 1 else "adversary"
    party_status = "democrat" if row['is_democrat'] == 1 else "republican"
    completion = f"{ally_status}, {party_status}"

    return {
        "messages": [
            {"role": "user", "content": row['phrase']},
            {"role": "assistant", "content": completion}
        ]
    }

# Convert DataFrame to the required format
chat_data = [create_chat_format(row) for _, row in df.iterrows()]

# Save to a JSONL file
with open('finetuning_data_chat.jsonl', 'w') as f:
    for entry in chat_data:
        f.write(json.dumps(entry) + '\n')

# Upload data to openai



In [ ]:
pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.4 MB/s eta 0:00:00


In [ ]:
def upload_data_to_open_ai(filename):
  client = OpenAI(api_key="xxxxx")

  return client.files.create(
    file=open(filename, "rb"),
    purpose="fine-tune"
  )

In [ ]:
upload_data_to_open_ai("finetuning_data_ally_adv_chat.jsonl")

FileObject(id='file-Oeov9PGWvVADGSbAKunB07Sq', bytes=43292, created_at=1716437517, filename='finetuning_data_ally_adv_chat.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="xxxxx")

client.files.create(
  file=open("finetuning_data_chat.jsonl", "rb"),
  purpose="fine-tune"
)

FileObject(id='file-NGCithb9s2z6iTxGFJNVqqDX', bytes=20092, created_at=1716417839, filename='finetuning_data_chat.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)

# Fine-tune the model

In [ ]:
def fine_tune_model(training_file):
  from openai import OpenAI
  client = OpenAI(api_key="xxxxx")

  return client.fine_tuning.jobs.create(
    training_file=training_file,
    model="gpt-3.5-turbo"
  )

In [ ]:
fine_tune_model("yyyyy")

FineTuningJob(id='ftjob-e7hGpuMS7FU14r0COSNE6WMo', created_at=1716437661, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-bwbLUhkkC3gTynzJ16LT3Bas', result_files=[], seed=1151039181, status='validating_files', trained_tokens=None, training_file='file-Oeov9PGWvVADGSbAKunB07Sq', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)

In [ ]:
# from openai import OpenAI
# client = OpenAI(api_key="xxxxx")

# client.fine_tuning.jobs.create(
#   training_file="file-NGCithb9s2z6iTxGFJNVqqDX",
#   model="gpt-3.5-turbo"
# )

FineTuningJob(id='ftjob-jgqtsC4PaXfNy63jKRkRImcp', created_at=1716417850, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-bwbLUhkkC3gTynzJ16LT3Bas', result_files=[], seed=521802189, status='validating_files', trained_tokens=None, training_file='file-NGCithb9s2z6iTxGFJNVqqDX', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)

# Monitor the fine-tuning job

In [ ]:
#e7hGpuMS7FU14r0COSNE6WMo
from openai import OpenAI
client = OpenAI(api_key="xxxxx")

# Retrieve the state of a fine-tune
model_state_info = client.fine_tuning.jobs.retrieve('ftjob-e7hGpuMS7FU14r0COSNE6WMo')
model_state_info

FineTuningJob(id='ftjob-e7hGpuMS7FU14r0COSNE6WMo', created_at=1716437661, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-3.5-turbo-0125:personal::9RuHtrJT', finished_at=1716438252, hyperparameters=Hyperparameters(n_epochs=3, batch_size=1, learning_rate_multiplier=2), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-bwbLUhkkC3gTynzJ16LT3Bas', result_files=['file-1f0mvsp98EF0w15dDBriv4Ka'], seed=1151039181, status='succeeded', trained_tokens=21849, training_file='file-Oeov9PGWvVADGSbAKunB07Sq', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)

In [ ]:
# from openai import OpenAI
# client = OpenAI(api_key="xxxxx")

# # Retrieve the state of a fine-tune
# model_state_info = client.fine_tuning.jobs.retrieve('ftjob-jgqtsC4PaXfNy63jKRkRImcp')

# Use a fine-tuned model

In [ ]:
model_state_info

FineTuningJob(id='ftjob-e7hGpuMS7FU14r0COSNE6WMo', created_at=1716437661, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-3.5-turbo-0125:personal::9RuHtrJT', finished_at=1716438252, hyperparameters=Hyperparameters(n_epochs=3, batch_size=1, learning_rate_multiplier=2), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-bwbLUhkkC3gTynzJ16LT3Bas', result_files=['file-1f0mvsp98EF0w15dDBriv4Ka'], seed=1151039181, status='succeeded', trained_tokens=21849, training_file='file-Oeov9PGWvVADGSbAKunB07Sq', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)

In [ ]:
fine_tuned_model_id = model_state_info.fine_tuned_model
fine_tuned_model_id

'ft:gpt-3.5-turbo-0125:personal::9RuHtrJT'

In [ ]:
org_id = model_state_info.organization_id
org_id

'org-bwbLUhkkC3gTynzJ16LT3Bas'

In [ ]:
model_id = "ft:gpt-3.5-turbo-0125:personal::9RuHtrJT"

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="xxxxx")

def get_prediction(prompt, model):


  completion = client.chat.completions.create(
    model=model,
    messages=[
      {"role": "system", "content": "You are a smart assistant who can read a phrase and know from the phrase whether Congress is talking about an ally nation or an adversary nation."},
      {"role": "user", "content": prompt},
    ]
  )

  print(completion.choices[0].message)
  return completion.choices[0].message




  # return {
  #       "messages": [
  #           {"role": "system", "content": "You are a smart assistant who can read a phrase and know from the phrase whether Congress is talking about an ally nation or an adversary nation."},
  #           {"role": "user", "content": user_message},
  #           {"role": "assistant", "content": completion}
  #       ]



In [ ]:
prompt = "This nation is a vital partner and friend of the United States" +  "Is the conversation around an ally nation or adversary nation?"
response = get_prediction(prompt, model_id)

ChatCompletionMessage(content='ally', role='assistant', function_call=None, tool_calls=None)


In [ ]:
response.content

'adversary, republican'

In [ ]:
fine_tuning.job.checkpoint('ftjob-jgqtsC4PaXfNy63jKRkRImcp')

NameError: name 'fine_tuning' is not defined

In [ ]:
import openai

openai.api_key = "xxxxx"


from openai import OpenAI
client = OpenAI()

client.files.create(
  file=open("mydata.jsonl", "rb"),
  purpose="fine-tune"
)



client = OpenAI()

client.fine_tuning.jobs.create(
  training_file="finetuning_data.jsonl",
  model="gpt-3.5-turbo"
)

# Monitor the fine-tuning job
fine_tune_id = response['id']

# Check the status
status = openai.FineTune.retrieve(fine_tune_id)
print(status)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Say this is a test",
        }
    ],
    model="gpt-3.5-turbo",
)

In [ ]:
response = openai.Completion.create(
  model="your-fine-tuned-model",
  prompt="This nation is a vital partner and friend of the United States",
  max_tokens=10
)

print(response.choices[0].text.strip())